# EYES-DEFY-ANEMIA -- Step 1: Measurement Harness

Pooled out-of-fold repeated stratified cross-validation for all 12 clean-data CNN combos.
**No model is changed and no intervention is applied** -- this run exists solely to replace a
statistically unusable estimator with a usable one, and to produce the frozen baseline that
Steps 2-5 are measured against.

**The problem being fixed.** The single 70/15/15 split leaves 14 India validation patients
(10 anemic / 4 healthy), so India AUC is computed over 10x4 = 40 discordant pairs -- a 95% CI
half-width of roughly +/-0.27. Under that noise floor the observed India AUC spread across the
12 combos (0.550-1.000) is not a ranking, and no later intervention could be shown to work.

**What this run produces.** 5-fold x 5-repeat CV over the 184-patient train+val pool, stratified
on the compound country x label key. India pairs go from **40 to 1,311**; Italy from **60 to 1,680**
(palpebral; forniceal_palpebral is 1,311 / 1,501 -- all 6 patients missing a forniceal crop are Italy).

**The 33-patient test split is sealed** and is asserted absent from every fold. It is spent exactly
once, in Step 6.

Runtime note: 12 combos x 25 fits. `sync_outputs()` runs after every combo, so an interrupted
session still yields a downloadable zip of everything completed so far.

**Companion notebook:** `step1-cv-harness-vit.ipynb` covers the remaining 6 transformer
combos (`swin_t`, `vit_b_16`, `vit_l_16` x both tissue types) separately -- run both, then merge
their downloaded `outputs/` locally before running `aggregate_baseline.py` to get the full
18-combo Step 1 baseline. This notebook alone produces a 12-combo (CNN-only) baseline.

## Setup

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first (this bit an earlier
# version of this notebook during interactive debugging).
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

In [ ]:
# Diagnostic only -- kept for visibility in the saved run log. The path used
# below is already confirmed correct, so this cell doesn't gate anything, but
# it's cheap and makes a future path change easy to spot in the output.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

In [ ]:
# Only packages actually missing from Kaggle's base image. Deliberately NOT
# `pip install -r requirements.txt` -- that file is pinned to the local
# Windows/CUDA 13.0 build and would try to reinstall Kaggle's own correctly
# configured GPU PyTorch with an incompatible build.
!pip install -q optuna albumentations

## Data

In [ ]:
import shutil
from pathlib import Path

# TODO: verify against cell 4's /kaggle/input listing before running -- this is a best-guess
# default following the established manivafapour33/<slug> pattern, NOT yet confirmed for the
# new clean-data dataset (see classification/.project_memory/kaggle/01_kaggle_notes.md).
SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset-clean")
DST_DIR = Path("classification/data/processed")

# Clear the destination first so this cell is fully idempotent -- a re-run (or
# the two messy copy attempts from the earlier notebook version) never leaves
# stale or duplicated content behind. DST_DIR always ends up as an exact,
# deterministic copy of SRC_DIR, nothing more.
shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

## Structural verification -- checkpoints 1-4 and 6

Trains nothing; runs in seconds. Fold geometry is where a silent error would be most damaging
and least visible: a leaked test patient or a fold holding only 2 India-healthy patients would
not raise, it would quietly yield a plausible-looking wrong number.

**This cell gates the run.** A non-zero exit means do not proceed to training.

In [ ]:
!python classification/step1_cv_harness/validate_harness.py

In [ ]:
# Locked hyperparameters, read straight out of each combo's own v2_clean study summary
# (never hand-transcribed). No re-tuning happens in Step 1 -- re-tuning inside the new
# protocol would confound 'better measurement' with 'better hyperparameters'.
!python classification/step1_cv_harness/run_cv_harness.py --list

## Output syncing

In [ ]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate classification/step1_cv_harness/outputs/ into
    /kaggle/working/outputs/ and re-zip to step1_cv_results_cnn.zip. Called after
    EVERY combo, not just at the end -- this is the longest job this project has
    attempted, so whatever has completed must always be downloadable."""
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    src = Path("classification/step1_cv_harness/outputs")
    if src.exists():
        shutil.copytree(src, results_dir, dirs_exist_ok=True)
    archive = shutil.make_archive("/kaggle/working/step1_cv_results_cnn", "zip", root_dir=str(results_dir))
    n = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n} files under {results_dir}, zipped to {archive}")


sync_outputs()  # picks up structural_verification.json; confirms the function works before training

## Training -- 12 combos, cheapest architecture first

25 fits per combo (5 folds x 5 repeats). Each fold trains on its own inner-split early stopping
and predicts once on its held-out outer fold; the outer fold is never consulted during training
or epoch selection. No checkpoints are written -- Step 1 needs predictions, not weights.

Budget fallback if the session is tight on time: add `--repeats 3` to any cell.

In [ ]:
# Step 1 -- 1/12: regnet_y_400mf_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo regnet_y_400mf_palpebral_v2_clean
sync_outputs()

In [ ]:
# Step 1 -- 2/12: regnet_y_400mf_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo regnet_y_400mf_forniceal_palpebral_v2_clean
sync_outputs()

In [ ]:
# Step 1 -- 3/12: mobilenet_v3_small_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo mobilenet_v3_small_palpebral_v2_clean
sync_outputs()

In [ ]:
# Step 1 -- 4/12: mobilenet_v3_small_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo mobilenet_v3_small_forniceal_palpebral_v2_clean
sync_outputs()

In [ ]:
# Step 1 -- 5/12: efficientnet_b0_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo efficientnet_b0_palpebral_v2_clean
sync_outputs()

In [ ]:
# Step 1 -- 6/12: efficientnet_b0_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo efficientnet_b0_forniceal_palpebral_v2_clean
sync_outputs()

In [ ]:
# Step 1 -- 7/12: resnet18_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo resnet18_palpebral_v2_clean
sync_outputs()

In [ ]:
# Step 1 -- 8/12: resnet18_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo resnet18_forniceal_palpebral_v2_clean
sync_outputs()

In [ ]:
# Step 1 -- 9/12: densenet121_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo densenet121_palpebral_v2_clean
sync_outputs()

In [ ]:
# Step 1 -- 10/12: densenet121_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo densenet121_forniceal_palpebral_v2_clean
sync_outputs()

In [ ]:
# Step 1 -- 11/12: convnext_tiny_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo convnext_tiny_palpebral_v2_clean
sync_outputs()

In [ ]:
# Step 1 -- 12/12: convnext_tiny_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo convnext_tiny_forniceal_palpebral_v2_clean
sync_outputs()

## Checkpoint 5 -- label-shuffle negative control

The cheapest available proof that the harness does not leak. Given this project's history with
silent data bugs (the white-background convention, v1 template matching), this is the check
least worth skipping.

- **`within_country`** permutes labels inside each country, preserving each country's label rate
  (India ~71.6% anemic, Italy ~18.9%) while destroying every per-patient association.
  Per-country AUC **must** collapse to 0.50. Overall AUC may legitimately stay above 0.50 --
  which is a direct empirical demonstration of the country-shortcut mechanism, since 62% of the
  overall AUC's pairs are cross-country.
- **`global`** destroys the country-label association too, so *every* AUC must collapse to 0.50.
  A stricter pure-leakage test that cannot demonstrate the mechanism.

In [ ]:
# Control A -- within-country permutation (leakage test + shortcut demonstration)
!python classification/step1_cv_harness/run_cv_harness.py \
    --combo mobilenet_v3_small_palpebral_v2_clean --shuffle-control within_country
sync_outputs()

In [ ]:
# Control B -- global permutation (strict leakage test: every AUC must reach chance)
!python classification/step1_cv_harness/run_cv_harness.py \
    --combo mobilenet_v3_small_palpebral_v2_clean --shuffle-control global
sync_outputs()

## Aggregate -- checkpoints 7, 8, 9

Writes the frozen Step 1 baseline (`outputs/baseline/step1_baseline.{csv,json,md}`) and evaluates
the remaining gates. Exits non-zero and refuses to declare Step 1 clear if any gate fails.

- **7 plausibility** -- pooled India AUC far outside what the single-split CIs already permitted
  means suspect the harness, not celebrate a discovery.
- **8 precision** -- India AUC 95% CI half-width <= 0.12. This is the actual pass/fail criterion:
  if it fails, Steps 3-5 cannot demonstrate anything and their success criteria must be
  renegotiated rather than quietly ignored.
- **9 artifact** -- the baseline file itself, recording seed, fold config and locked
  hyperparameters so later steps can run a valid *paired* comparison against it.

**Note: run standalone, this aggregates only the 12 CNN combos present in `outputs/` at this
point** -- it is a valid partial check (all 4 gates still apply and mean something), but it is
not the final Step 1 baseline. Get the full 18-combo baseline by downloading both this notebook's
and `step1-cv-harness-vit.ipynb`'s output zips, merging their `outputs/` folders locally into one
`classification/step1_cv_harness/outputs/`, and re-running `aggregate_baseline.py` there --
combo discovery is dynamic (globs `outputs/*/cv_metrics.json`), so no code change is needed.

In [ ]:
!python classification/step1_cv_harness/aggregate_baseline.py
sync_outputs()

In [ ]:
from pathlib import Path

report = Path('classification/step1_cv_harness/outputs/baseline/step1_baseline.md')
print(report.read_text(encoding='utf-8') if report.exists() else 'Baseline not produced -- check the aggregate cell above.')

## Done -- what to download

`/kaggle/working/step1_cv_results_cnn.zip` contains, per combo: `oof_predictions.csv` (one held-out
probability per patient per repeat), `fold_manifest.json` (the exact fold assignments, which
Steps 3-5 **must** reuse so their comparison against this baseline can be paired), and
`cv_metrics.json`. Plus `baseline/` and `structural_verification.json`.

Everything is small text -- no checkpoints, by design.

**Next step after both notebooks finish:** download this zip and `step1-cv-harness-vit.ipynb`'s
zip, extract both `outputs/` into the same local `classification/step1_cv_harness/outputs/`, then
run `aggregate_baseline.py` once locally for the combined 18-combo baseline.

In [ ]:
from pathlib import Path

print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.3f} MB)')

zip_path = Path('/kaggle/working/step1_cv_results_cnn.zip')
print(f'\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)')